# 회사별 권리 일정 달력

배당 기준일·권리락일·지급일과 주총 개최일을 SEIBRO WebSquare에서 조회합니다.
일정일자와 권리기준일을 함께 보면 한 권리에 연결된 여러 일정을 구별할 수 있습니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/seibro-api/blob/main/scenarios/web/02_company_schedules.ipynb)

## 0. 준비
이 시나리오는 WebSquare 공개 화면을 조회하며 API 키가 필요 없습니다. 아래 두 셀을 실행해 패키지를 설치하고 실행 환경을 준비하세요.

In [1]:
# Colab에서만: 패키지 설치 (로컬은 이 레포를 pip install -e . 로 설치해 두면 됨)
import sys
if "google.colab" in sys.modules:
    !pip -q install "git+https://github.com/beaten-by-the-market/seibro-api.git" koreanize-matplotlib

In [2]:
SECRETS = []
# ── SEIBRO 공통 셀 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 아래 SECRETS 이름으로 값을 넣고 "노트북 액세스"를 켜세요.
#        SEIBRO_API_KEY = 예탁결제원 SEIBRO Open API 인증키 (https://seibro.or.kr/OpenPlatform)
#        DART_API_KEY = OpenDART 인증키 (DART 데이터를 쓰는 노트북만)
#        KRX_ID / KRX_PW = KRX 정보데이터시스템 계정 (KRX 데이터를 쓰는 노트북만)
# 로컬: 환경변수 또는 작업 폴더(레포 루트)의 .env 를 읽습니다.
# 키·비밀번호 값은 어디에도 출력하지 않습니다. 있는지 여부만 보여 줍니다.
import os, sys, warnings
from datetime import date, datetime, timedelta

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

if IN_COLAB and SECRETS:
    from google.colab import userdata
    for _name in SECRETS:  # noqa: F821  (빌더가 이 셀 맨 위에 SECRETS 목록을 넣음)
        if os.environ.get(_name):
            continue
        try:
            _v = userdata.get(_name)
        except Exception:
            _v = None
        if not _v:
            from getpass import getpass
            _v = getpass(f"{_name}: ")
        os.environ[_name] = _v.strip()
elif SECRETS:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))

_missing = [
    n for n in SECRETS  # noqa: F821
    if not os.environ.get(n) and (IN_COLAB or not n.startswith("KRX_"))
]
if _missing:
    raise RuntimeError(f"{_missing} 가 없습니다. Colab Secrets 또는 .env 에 넣어 주세요.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def won(v, digits=1):
    """원 → '1.2조' / '5,027억' / '3,400만'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


def shares(v):
    """주식수 → '23,607,980주'."""
    return "-" if v is None or pd.isna(v) else f"{int(v):,}주"


# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

def _status(n):
    if os.environ.get(n):
        return "있음"
    # 로컬에서 krx-data-api를 편집 설치(pip install -e)했다면 그 레포의 .env를 패키지가 직접 읽는다
    return "없음" if IN_COLAB or not n.startswith("KRX_") else "krx-data-api 자체 .env 사용"


credential_status = ", ".join(f"{n}: {_status(n)}" for n in SECRETS)
print("SEIBRO 준비 완료", "(Colab)" if IN_COLAB else "(로컬)",
      "| " + (credential_status or "웹 조회 · 키 필요 없음"))

SEIBRO 준비 완료 (로컬) | 웹 조회 · 키 필요 없음


In [3]:
from IPython.display import display

from seibro_api import get_company_schedules

STOCK_CODE = "005930"  # 삼성전자
START_DT = "20260101"
END_DT = "20260630"
REASON_CODE = ""  # 전체. 배당/분배만 보려면 "103"

In [4]:
events = get_company_schedules(
    STOCK_CODE, START_DT, END_DT, reason_code=REASON_CODE, save_csv=False, use_openapi=False,
)
if events.empty:
    print("조회된 권리 일정이 없습니다.")
else:
    counts = events.groupby(["사유", "일정종류"], dropna=False).size().rename("건수").reset_index()
    display(counts)
    display(events.filter(items=["일정일자", "일정종류", "사유", "사유상세", "기준일", "일정종료일"]))


[Seibro] 기간별 주요일정 조회
  -> 삼성전자 (005930, 고객번호: 593)
  기간: 20260101 ~ 20260630


  -> 32건 수집 완료


,사유,일정종류,건수
0,기타,기준일,4
1,기타,명부폐쇄기간,4
2,기타,명세접수시한,4
3,기타,명세통지시한,4
4,배당/분배,권리락일,2
5,배당/분배,기준일,2
6,배당/분배,명부폐쇄기간,2
7,배당/분배,명세접수시한,1
8,배당/분배,명세통지시한,1
9,배당/분배,배당금지급일(1차),2


,일정일자,일정종류,사유,사유상세,기준일,일정종료일
0,20260101,명부폐쇄기간,배당/분배,배당/분배(현금배당),20251231,20260131
1,20260101,명부폐쇄기간,정기총회,정기총회,20251231,20260131
2,20260130,기준일,기타,기타,20260130,20260130
3,20260131,명부폐쇄기간,기타,기타,20260130,
4,20260204,명세접수시한,기타,기타,20260130,20260204
5,20260209,명세통지시한,기타,기타,20260130,20260209
6,20260227,기준일,기타,기타,20260227,20260227
7,20260228,명부폐쇄기간,기타,기타,20260227,
8,20260303,발송통지시한,정기총회,정기총회,20251231,20260303
9,20260305,명세접수시한,기타,기타,20260227,20260305


각 행은 일정 하나입니다. 같은 기준일에 연결된 기준일·지급일·권리락일을 각각 집계합니다.
주총 안건이 필요하면 [공식 API 주총 조회](../openapi/02_shareholder_meetings.md)를 사용합니다.